In [ ]:
import kagglehub

In [ ]:
path=kagglehub.dataset_download('sriramr/fruits-fresh-and-rotten-for-classification')

In [ ]:
import tensorflow as tf
import os

In [ ]:
train_path=os.path.join(path,'dataset','train')

In [ ]:
val_path=os.path.join(path,'dataset','test')

In [ ]:
img_height=224
img_width=224
batch_size=32

In [ ]:
train_ds=tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(img_height,img_width),
    batch_size=batch_size,
    seed=123,
    subset='training',
    validation_split=0.2
)

In [ ]:
val_ds=tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(img_height,img_width),
    batch_size=batch_size,
    seed=123,
    subset='validation',
    validation_split=0.2
)

In [ ]:
original_class_name=train_ds.class_names

In [ ]:
original_class_name

In [ ]:
def normalize(image,label):
    image=tf.cast(image/255.0,tf.float32)
    return image,label

In [ ]:
train_ds=train_ds.map(normalize)
val_ds=val_ds.map(normalize)

In [ ]:
def map_labels_to_binary(image, label):
    new_label = tf.cast(label >= len(original_class_name) // 2, tf.float32)
    return image, new_label

In [ ]:
train_ds_binary = train_ds.map(map_labels_to_binary)
val_ds_binary = val_ds.map(map_labels_to_binary)

In [ ]:
data_augmentation=tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2)
])

In [ ]:
def augment(image,label):
  image=data_augmentation(image)
  return image,label

In [ ]:
train_ds_augmented=train_ds.map(augment)

In [ ]:
model=tf.keras.Sequential([
    tf.keras.layers.Conv2D(32,(3,3),activation='relu',input_shape=(img_height,img_width,3)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D((2,2)),
    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D((2,2)),
    tf.keras.layers.Conv2D(128,(3,3),activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64,activation='relu'),
    tf.keras.layers.Dense(1,activation='sigmoid')
])

In [ ]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
# history=model.fit(train_ds_augmented,validation_data=val_ds_binary,epochs=20,callbacks=[early_stopping])

In [ ]:
test_path=os.path.join(path,'dataset','test')

In [ ]:
test_ds=tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(img_height,img_width),
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
test_ds=test_ds.map(normalize)
test_ds=test_ds.map(map_labels_to_binary)

In [ ]:
p='fruits_classification_model.keras'
p2='fruits_classification_model.h5'

model.save(p)
model.save(p2)

In [ ]:
import pickle

try:
    with open('fruits_classification_model.keras', 'rb') as f:
        Model = pickle.load(f)
    print("Model loaded successfully using pickle!")
except Exception as e:
    print(f"Could not load model using pickle: {e}")
    print("Using the in-memory 'model' object instead for evaluation.")
    Model = model # Fallback to the in-memory model if pickle fails

loss,accuracy=Model.evaluate(test_ds)

test_loss=loss
test_accuracy=accuracy

print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")